# 02 — Bronze (Auto Loader → VARIANT Delta)

Reads every JSON file landed in the Volume by notebook 01 into two
**VARIANT-typed** bronze tables. One run is `availableNow=True`, so re-runs
just pick up files added since the last execution (idempotent).

| Bronze table | Source folder in Volume                | Grain |
|--------------|----------------------------------------|-------|
| `bronze_artists` | `artists/{handle}.json`            | One per artist |
| `bronze_team`    | `team/{handle}/{role}.json`        | One per (artist, role) |

Every bronze row carries:
- `data VARIANT` — the full raw payload, navigable with `data:field::type`
- `_ingestion_timestamp` — when the row landed in bronze
- `_source_file` — the Volume path it came from

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

In [ ]:
UC_CATALOG    = os.getenv('UC_CATALOG', 'alexander_booth')
UC_SCHEMA     = os.getenv('UC_SCHEMA',  'rostr_music_industry')
BRONZE_SCHEMA = f'{UC_SCHEMA}_bronze'
VOLUME_PATH   = f'/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/raw_data'

ENTITIES = [
    # (bronze_table_suffix, glob_path under raw_data)
    ('artists', 'artists/*.json'),
    ('team',    'team/*/*.json'),
]
print(f'Volume: {VOLUME_PATH}')
print(f'Bronze : {UC_CATALOG}.{BRONZE_SCHEMA}')
print(f'Entities: {ENTITIES}')

In [ ]:
for entity, glob in ENTITIES:
    source_dir      = f"{VOLUME_PATH}/{glob.rsplit('/', 1)[0]}/"
    glob_filter     = glob.rsplit('/', 1)[1]
    checkpoint_path = f'{VOLUME_PATH}/_checkpoints/{entity}'
    target_table    = f'{UC_CATALOG}.{BRONZE_SCHEMA}.bronze_{entity}'

    print(f'\n[{entity}] -> {target_table}')
    print(f'  source: {source_dir}{glob_filter}')

    try:
        (
            spark.readStream
            .format('cloudFiles')
            .option('cloudFiles.format', 'json')
            .option('cloudFiles.inferColumnTypes', 'true')
            .option('cloudFiles.schemaLocation', checkpoint_path + '/_schema')
            .option('cloudFiles.schemaEvolutionMode', 'rescue')
            .option('pathGlobFilter', glob_filter)
            .option('multiLine', 'true')
            .load(source_dir)
            .selectExpr(
                'PARSE_JSON(TO_JSON(STRUCT(*))) AS data',
                'current_timestamp() AS _ingestion_timestamp',
                '_metadata.file_path AS _source_file',
            )
            .writeStream
            .format('delta')
            .outputMode('append')
            .option('checkpointLocation', checkpoint_path)
            .trigger(availableNow=True)
            .toTable(target_table)
        ).awaitTermination()
        n = spark.table(target_table).count()
        print(f'  done ({n:,} rows total)')
    except Exception as e:
        # Empty source folders are common during first-time partial ingests.
        print(f'  skipped: {str(e)[:160]}')

## Verify

In [ ]:
print('=' * 60)
print('BRONZE LAYER SUMMARY')
print('=' * 60)
for entity, _ in ENTITIES:
    table = f'{UC_CATALOG}.{BRONZE_SCHEMA}.bronze_{entity}'
    try:
        n = spark.table(table).count()
        print(f'  {table:<70s}  {n:>10,} rows')
    except Exception as e:
        print(f'  {table:<70s}  (missing — {str(e)[:60]})')

print('\nSample bronze_artists (raw VARIANT navigation):')
spark.sql(f"""
    SELECT data:rostrId::string  AS rostr_id,
           data:name::string     AS name,
           data:artistType::string AS artist_type,
           data:spMetric::bigint AS spotify_followers,
           _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.bronze_artists
    LIMIT 5
""").show(truncate=False)

print('Sample bronze_team (raw VARIANT navigation):')
spark.sql(f"""
    SELECT REGEXP_EXTRACT(_source_file, 'team/([^/]+)/([^.]+)\\\\.json', 1) AS artist_handle,
           REGEXP_EXTRACT(_source_file, 'team/([^/]+)/([^.]+)\\\\.json', 2) AS role,
           SIZE(FROM_JSON(TO_JSON(data), 'ARRAY<STRUCT<company:STRUCT<name:STRING>>>')) AS company_count,
           _ingestion_timestamp
    FROM {UC_CATALOG}.{BRONZE_SCHEMA}.bronze_team
    LIMIT 5
""").show(truncate=False)